In [0]:
from pyspark.sql import functions as F

########Generate OrderLine records

In [0]:
from pyspark.sql import functions as F

SEED = 20261022
ORDER_LINE_COUNT = 8000

order_lines = spark.range(1, ORDER_LINE_COUNT + 1)

# Generate unique order line IDs
order_lines = order_lines.withColumn(
    "order_line_id",
    F.concat(
        F.lit("OL-"),
        F.lpad(F.col("id").cast("string"), 8, "0")
    )
)

# Assign valid order IDs
order_lines = order_lines.withColumn(
    "order_number",
    F.pmod(
        F.xxhash64("order_line_id", F.lit(SEED)),
        F.lit(4000)
    ) + 1
)

order_lines = order_lines.withColumn(
    "order_id",
    F.concat(
        F.lit("ORD-2026-"),
        F.lpad(F.col("order_number").cast("string"), 6, "0")
    )
)

# Assign valid product IDs
order_lines = order_lines.withColumn(
    "product_number",
    F.pmod(
        F.xxhash64("order_line_id", F.lit("product")),
        F.lit(20)
    ) + 1
)

order_lines = order_lines.withColumn(
    "product_id",
    F.concat(
        F.lit("PRD-"),
        F.lpad(F.col("product_number").cast("string"), 6, "0")
    )
)

# Generate quantity
order_lines = order_lines.withColumn(
    "quantity",
    (F.pmod(F.xxhash64("order_line_id", F.lit("quantity")), F.lit(5)) + 1).cast("int")
)

# Get actual product prices from Product Master
order_lines = order_lines.join(
    spark.read.table("product_master").select("product_id", "unit_price"),
    on="product_id",
    how="inner"
)

# Generate discount percentage
order_lines = order_lines.withColumn(
    "discount",
    F.when(F.pmod(F.xxhash64("order_line_id", F.lit("discount")), F.lit(100)) < 50, 0)
     .when(F.pmod(F.xxhash64("order_line_id", F.lit("discount")), F.lit(100)) < 80, 5)
     .otherwise(10)
)

# Calculate amounts
order_lines = order_lines.withColumn(
    "gross_amount",
    F.round(F.col("quantity") * F.col("unit_price"), 2)
)

order_lines = order_lines.withColumn(
    "discount_amount",
    F.round(F.col("gross_amount") * F.col("discount") / 100, 2)
)

order_lines = order_lines.withColumn(
    "line_total",
    F.round(F.col("gross_amount") - F.col("discount_amount"), 2)
)

order_line_df = order_lines.select(
    "order_line_id",
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "discount",
    "gross_amount",
    "discount_amount",
    "line_total"
)

display(order_line_df.limit(10))

order_line_id,order_id,product_id,quantity,unit_price,discount,gross_amount,discount_amount,line_total
OL-00000001,ORD-2026-000298,PRD-000012,1,30000,0,30000,0.0,30000.0
OL-00000002,ORD-2026-000650,PRD-000011,2,6000,5,12000,600.0,11400.0
OL-00000003,ORD-2026-001156,PRD-000009,1,25000,10,25000,2500.0,22500.0
OL-00000004,ORD-2026-002793,PRD-000014,1,200,0,200,0.0,200.0
OL-00000005,ORD-2026-003102,PRD-000013,2,1000,10,2000,200.0,1800.0
OL-00000006,ORD-2026-002959,PRD-000007,5,3000,5,15000,750.0,14250.0
OL-00000007,ORD-2026-000237,PRD-000011,3,6000,0,18000,0.0,18000.0
OL-00000008,ORD-2026-001683,PRD-000010,2,3500,5,7000,350.0,6650.0
OL-00000009,ORD-2026-000396,PRD-000009,2,25000,10,50000,5000.0,45000.0
OL-00000010,ORD-2026-003757,PRD-000009,4,25000,5,100000,5000.0,95000.0


######Validate OrderLine records


In [0]:
#Check total records:
print("Total OrderLines:", order_line_df.count())

Total OrderLines: 8000


In [0]:
#Check duplicate IDs:
order_line_df.groupBy("order_line_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+-------------+-----+
|order_line_id|count|
+-------------+-----+
+-------------+-----+



In [0]:
#Check invalid order references:
invalid_orders = order_line_df.join(
    spark.read.table("order_master").select("order_id"),
    on="order_id",
    how="left_anti"
)

print("Invalid Order IDs:", invalid_orders.count())

Invalid Order IDs: 0


In [0]:
#Check invalid product references:
invalid_products = order_line_df.join(
    spark.read.table("product_master").select("product_id"),
    on="product_id",
    how="left_anti"
)

print("Invalid Product IDs:", invalid_products.count())

Invalid Product IDs: 0


In [0]:
#Check financial calculations
order_line_df.select(
    F.sum("gross_amount").alias("Total Gross Amount"),
    F.sum("discount_amount").alias("Total Discount"),
    F.sum("line_total").alias("Total Net Amount")
).show()

+------------------+--------------+----------------+
|Total Gross Amount|Total Discount|Total Net Amount|
+------------------+--------------+----------------+
|         201389700|     6763822.5|   1.946258775E8|
+------------------+--------------+----------------+



In [0]:
#Save the table
order_line_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("order_line")

In [0]:
%sql
SELECT * FROM order_line LIMIT 10;

order_line_id,order_id,product_id,quantity,unit_price,discount,gross_amount,discount_amount,line_total
OL-00002001,ORD-2026-003178,PRD-000009,3,25000,5,75000,3750.0,71250.0
OL-00002002,ORD-2026-002568,PRD-000002,5,25000,0,125000,0.0,125000.0
OL-00002003,ORD-2026-000506,PRD-000018,3,350,5,1050,52.5,997.5
OL-00002004,ORD-2026-003481,PRD-000017,3,250,5,750,37.5,712.5
OL-00002005,ORD-2026-003335,PRD-000013,3,1000,0,3000,0.0,3000.0
OL-00002006,ORD-2026-000536,PRD-000019,1,2000,5,2000,100.0,1900.0
OL-00002007,ORD-2026-002924,PRD-000006,4,1800,0,7200,0.0,7200.0
OL-00002008,ORD-2026-001498,PRD-000001,1,65000,0,65000,0.0,65000.0
OL-00002009,ORD-2026-003068,PRD-000012,4,30000,10,120000,12000.0,108000.0
OL-00002010,ORD-2026-000650,PRD-000007,2,3000,0,6000,0.0,6000.0
